In [ ]:
# imports

import os
import sys
from pathlib import Path
import json
import traceback
import re
from dotenv import load_dotenv
import csv
import random
import unicodedata
from collections import defaultdict, OrderedDict
import pandas as pd

from entities.document import Document
from entities.frames import *  # full coverage
import entities.frames as frames

load_dotenv('.env')
FIGMA_TOKEN = os.getenv('FIGMA_TOKEN')
FIGMA_DOCUMENT_ID1 = os.getenv('FIGMA_DOCUMENT_ID1')
FIGMA_DOCUMENT_ID2 = os.getenv('FIGMA_DOCUMENT_ID2')

page_list1=["Module 1","Module 2", "Module 3","Module 4","Module 9"]
page_list2=["Module 5", "Module 6","Module 7","Module 8"]
os.environ["FIGMA_API_KEY"] = FIGMA_TOKEN

In [ ]:
PHOTOBANK_CSV_PATH = "../02_Inputs/data/photobank.csv"

def load_photobank_images(csv_path=PHOTOBANK_CSV_PATH):
    images = []
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            images.append(row['file_path'])
    return images

PHOTOBANK_IMAGES = load_photobank_images()

In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

In [ ]:
##Generate english module structure

import pandas as pd
import json
import re
from pathlib import Path

# Utility functions
def normalize(text):
    return re.sub(r"[\u202f\xa0]", " ", str(text)).strip()

def clean_title(text, prefix=None):
    text = normalize(text)
    return re.sub(prefix, "", text).strip() if prefix else text

# Load data
lessons = pd.read_excel("../02_Inputs/data/SEA Table of Contents.xlsx","Lessons").fillna("")
metadata = pd.read_excel("../02_Inputs/data/SEA Table of Contents.xlsx","Chapters").fillna("")

# Preprocess lesson data
lessons = lessons[lessons["Lesson Number"].notna()]
lessons["Lesson Number"] = lessons["Lesson Number"].astype(str)
lessons["Module Number"] = lessons["Lesson Number"].apply(lambda x: x.split(".")[0])
lessons["Chapter Number"] = lessons["Lesson Number"].apply(lambda x: ".".join(x.split(".")[:2]))
lessons["Module Title"] = lessons["Module Title"].ffill()
lessons["Chapter Title"] = lessons["Chapter Title"].ffill()

# Build lookup tables
module_meta = (
    metadata[metadata["type"] == "module"]
    .set_index("module_id")
    .rename_axis(None)  # optional: cleaner index
)
module_meta.index = module_meta.index.astype(str)
module_meta = module_meta.to_dict("index")

chapter_meta = (
    metadata[metadata["type"] == "chapter"]
    .set_index("chapter_id")
    .rename_axis(None)
)
chapter_meta.index = chapter_meta.index.astype(str)
chapter_meta = chapter_meta.to_dict("index")
lesson_desc = lessons.groupby("Lesson Number")["Description"].first().fillna("").to_dict()


# Generate module structure
structure = {"modules": []}


for module_id, mdf in lessons.groupby("Module Number"):
    module_id = str(module_id)
    meta = module_meta.get(module_id, {})
    module_title = clean_title(mdf["Module Title"].iloc[0], r"^Module\s*\d+:?\s*")

    module = {
        "id": module_id,
        "title": module_title,
        "color": meta.get("color", ""),
        "icon": f"https://sehseadata.blob.core.windows.net/images/Icons/Modules/module{module_id}.svg",
        "image": {
            "src": f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Modules/module{module_id}.webp",
            "caption": ""
        },
        "description": meta.get("description", ""),
        "aiPrompt": meta.get("description", ""),##update this
        "percentComplete": "0",
        "estimatedTime": meta.get("estimatedTime", ""),
        "chapters": []
    }

    # Module intro (add preface only for Module 1)
    intro_lessons = []
    
    if module_id == "1":
        intro_lessons.append({
            "type": "lesson",
            "title": f"Module {module_id} Preface",
            "id": f"{module_id}.0.-1",
            "progress": "not_started",
            "description": "Preface to the module and overview of what’s ahead.",
            "aiPrompt":meta.get("description", ""),##update this
            "image": {
                "src": f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Modules/module{module_id}.webp",
                "caption": ""
            }
        })
    
    intro_lessons.append({
        "type": "lesson",
        "title": f"Module {module_id} Introduction",
        "id": f"{module_id}.0.0",
        "progress": "not_started",
        "description": f"Introduction and learning objectives for Module {module_id}",
        "aiPrompt":meta.get("description", ""),##update this
        "image": {
            "src": f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Modules/module{module_id}.webp",
            "caption": ""
        }
    })
    
    module["chapters"].append({
        "id": f"{module_id}.0",
        "title": f"Module {module_id} Introduction",
        "icon": module["icon"],
        "lessons": intro_lessons
    })


    for i, (chapter_id, cdf) in enumerate(mdf.groupby("Chapter Number")):
        chapter_id = str(chapter_id)
        chapter_meta_entry = chapter_meta.get(chapter_id, {})
        chapter_title = clean_title(cdf["Chapter Title"].iloc[0], r"^Chapter\s*\d+:?\s*")

        chapter = {
            "id": chapter_id,
            "title": f"Chapter {i+1}: <strong>{chapter_title}</strong>",
            "icon": f"https://sehseadata.blob.core.windows.net/images/Icons/Chapters/chapter{chapter_id}.svg",
            "image": {
                "src": f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Chapters/chapter{chapter_id}.webp",
                "caption": ""
            },
            "description": chapter_meta_entry.get("description", ""),
            "aiPrompt":chapter_meta_entry.get("description", ""),##update this
            "estimatedTime": chapter_meta_entry.get("estimatedTime", ""),
            "lessons": []
        }

        # Chapter intro
        chapter["lessons"].append({
            "type": "lesson",
            "title": f"Chapter {i+1} Introduction",
            "id": f"{chapter_id}.0",
            "progress": "not_started"
        })


        # Real lessons
        for _, row in cdf.drop_duplicates(subset="Lesson Number").iterrows():
            lesson_id = row["Lesson Number"]
            if pd.isna(row["Lesson Title"]) or lesson_id not in lesson_desc:
                continue  # skip if no valid title or no metadata
        
            lesson_title = f"Lesson {lesson_id[-1]}: <strong>{clean_title(row['Lesson Title'])}</strong>"
            chapter["lessons"].append({
                "type": "lesson",
                "id": lesson_id,
                "title": lesson_title,
                "progress": "not_started",
                "description": lesson_desc.get(lesson_id, ""),
                "aiPrompt":lesson_desc.get(lesson_id, ""),##update this
                "frame": {
                    "src": f"https://sehseadata.blob.core.windows.net/images/Lessons/frame/lesson{lesson_id.replace('.', '')}_frame.webp",
                    "caption": ""
                },
                "image": {
                    "src": f"https://sehseadata.blob.core.windows.net/images/Lessons/image/lesson{lesson_id.replace('.', '')}_image.webp",
                    "caption": ""
                }
            })


        # Chapter outro
        chapter["lessons"].append({
            "type": "lesson",
            "title": f"Chapter {i+1} Outro",
            "id": f"{chapter_id}.-1",
            "progress": "not_started",
            "description": "Outro of the chapter and wrap-up.",
            "aiPrompt":chapter_meta_entry.get("description", ""),##update this
            "image": {
                "src": f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Chapters/chapter{chapter_id}.webp",
                "caption": ""
            }
        })

        # 4) Build tooltip now that all lessons are in place
        base_desc = chapter_meta_entry.get("description", "")
        
        lesson_lines = []
        for entry in chapter["lessons"]:
            lid = entry["id"]
            # Skip intro (ends with ".0") and outro (ends with ".-1")
            if lid.endswith(".0") or lid.endswith(".-1"):
                continue
            # Build a line like: "Lesson 1: <strong>Title</strong>."
            idx = len(lesson_lines) + 1
            lesson_lines.append(f"{entry['title']}.")
        
        if lesson_lines:
            # Join each line with a "<br>" prefix
            formatted_lessons = "<br> ".join(lesson_lines)
            chapter["tooltip"] = f"{base_desc} <br> {formatted_lessons}"
        else:
            chapter["tooltip"] = base_desc


        module["chapters"].append(chapter)
    
    # Module-level quiz lesson (e.g., 1.-1.0)
    quiz_lesson = {
        "type": "lesson",
        "title": f"Module {module_id} Quiz",
        "id": f"{module_id}.-1.0",
        "progress": "not_started",
        "description": "Test your understanding of all lessons in this module.",
        "aiPrompt": meta.get("description", ""),##update this
        "image": {
            "src": f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Modules/module{module_id}.webp",
            "caption": ""
        }
    }
    
    # Module outro chapter
    module_outro_chapter = {
        "id": f"{module_id}.-1",
        "title": f"Module {module_id} Outro",
        "icon": "https://sehseadata.blob.core.windows.net/images/Icons/module-outro.svg",
        "description": meta.get("description", ""),
        "aiPrompt": meta.get("description", ""),##update this
        "lessons": [
            quiz_lesson,
            {
                "type": "lesson",
                "title": f"Module {module_id} Outro",
                "id": f"{module_id}.-1.-1",
                "progress": "not_started",
                "image": {
                    "src": f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Modules/module{module_id}.webp",
                    "caption": ""
                }
            }
        ]
    }
    
    module["chapters"].append(module_outro_chapter)

    structure["modules"].append(module)

# Save JSON
with open("../03_Outputs/SEA_Modules/en/module_structure.json", "w", encoding="utf-8") as f:
    json.dump(structure, f, indent=2, ensure_ascii=False)

print("✅ module_structure.json generated successfully.")


In [ ]:
clearedInfographicsList=['M1_C1_3','M5_C3_4', 'M4_C1_6', 'M3_C2_2', 'M8_C2_5', 'M5_C1_1', 'M3_C2_5', 'M5_C1_2', 'M1_C1_8', 'M3_C2_4', 'M8_C4_1', 'M8_C1_2', 'M3_C1_1', 'M2_C1_1', 'M8_C2_10', 'M4_C2_7', 'M8_C1_1', 'M5_C2_5', 'M5_C4_3', 'M3_C1_3', 'M2_C1_3', 'M7_C1_9', 'M4_C2_4', 'M8_C1_4', 'M3_C3_5', 'M3_C1_7', 'M8_C2_16', 'M4_C2_1', 'M4_C2_3', 'M5_C2_3', 'M3_C3_6', 'M2_C3_6', 'M5_C4_5', 'M3_C1_4', 'M2_C1_4', 'M2_C1_5', 'M3_C1_5', 'M8_C2_15', 'M2_C3_7', 'M7_C1_10', 'M7_C1_2', 'M2_C1_9', 'M5_C4_8', 'M7_C3_1', 'M6_C3_1', 'M2_C1_13', 'M6_C1_3', 'M1_C2_4', 'M8_C1_8', 'M2_C1_11', 'M7_C3_3', 'M6_C3_2', 'M2_C3_8', 'M3_C3_8', 'M8_C1_9', 'M1_C2_2', 'M6_C3_5', 'M7_C3_4', 'M6_C1_6', 'M1_C2_1', 'M7_C2_3']

In [ ]:


# ===============================
# Constants & Global Variables
# ===============================
FIGMA_FOLDER = "../02_Inputs/figma_jsons"
OUTPUT_DIR_BASE = "../03_Outputs/SEA_Modules/en"

LESSON_NEXT_MAP = {}

ORDER_ISSUE_CATEGORIES = {
    "missing_cover": [],
    "wrong_second": [],
    "missing_key_takeaways": [],
    "has_embed": [] 

}

# ========
# Entity keywords for AI tags
# ========

ENTITY_CSV_PATH = "../02_Inputs/entities.csv"
AI_TAG_PROMPTS = [
    "What does {keyword} mean in this context?",
    "Explain further about {keyword}",
    "Provide an example of {keyword}",
    "Why is {keyword} important here?",
    "Summarize the concept of {keyword}",
    "How does {keyword} apply to development?",
    "What are the implications of {keyword}?",
    "Describe the challenges with {keyword}",
    "What are the benefits of {keyword}?",
    "How is {keyword} evolving in practice?"
]

AI_TAG_PROMPTS = [
    "What is {keyword}?",
    "Tell me more about {keyword}.",
    "Explain {keyword}.",
    "Share details about {keyword}.",
    "Clarify {keyword}.",
    "Provide more info on {keyword}."
]

ENTITY_KEYWORDS = set(
    pd.read_csv(ENTITY_CSV_PATH)["Entity"].dropna().str.strip().str.lower().tolist()
)

def tag_ai_keywords_in_text_segments(segments):
    """
    Tags a single keyword per eligible 'text' segment using <ai> markup.
    Ensures: one keyword per frame, and each keyword tagged only once per lesson.
    """
    used_keywords = set()
    sorted_keywords = sorted(ENTITY_KEYWORDS, key=lambda x: -len(x))  # match longest first

    for segment in segments:
        if segment.get("template_id") != "text":
            continue

        content = segment.get("content", {})
        text_elements = content.get("text_elements", [])
        if not isinstance(text_elements, list):
            continue

        tagged_this_frame = False
        for te in text_elements:
            if tagged_this_frame:
                break
            te_content = te.get("content", {})
            original_text = te_content.get("text", "")
            if not isinstance(original_text, str):
                continue

            lowered_text = original_text.lower()

            for keyword in sorted_keywords:
                if keyword in used_keywords:
                    continue
                if re.search(rf'\b{re.escape(keyword)}\b', lowered_text):
                    prompt = random.choice(AI_TAG_PROMPTS).format(keyword=keyword)
                    ai_tag = f'<ai prompt="{prompt}">{keyword} 💬️</ai>'
                    new_text = re.sub(rf'\b({re.escape(keyword)})\b', ai_tag, original_text, count=1, flags=re.IGNORECASE)

                    if new_text != original_text:
                        te["content"]["text"] = new_text
                        used_keywords.add(keyword)
                        tagged_this_frame = True
                        break

    return segments




# ===============================
# Figma Document Caching Helpers
# ===============================
def save_figma_to_json(figma_document, filename):
    os.makedirs(FIGMA_FOLDER, exist_ok=True)
    with open(os.path.join(FIGMA_FOLDER, filename), 'w', encoding='utf-8') as f:
        json.dump(figma_document.model_dump(), f, indent=2)

def load_cached_figma_page(filename, target_page_name):
    """
    Loads a cached Figma document JSON file and returns the page that matches target_page_name.
    """
    file_path = os.path.join(FIGMA_FOLDER, filename)
    if not Path(file_path).exists():
        raise ValueError(f"No cached Figma document found in {file_path}. Set redownload=True to download.")
    print("Loading cached Figma document...")
    with open(file_path, 'r', encoding='utf-8') as f:
        figma_data = json.load(f)
    figma_document = Document.model_validate(figma_data)
    try:
        page = next(
            page for page in figma_document.children
            if page.type == "CANVAS" and page.name == target_page_name
        )
        print(f"Found page: {target_page_name}")
        return page
    except StopIteration:
        raise ValueError(f"Page '{target_page_name}' not found in the document.")


# ===============================
# Section & Lesson Extraction
# ===============================
def get_valid_sections(page):
    """
    Filters the sections on a given page returning only those matching allowed section names.
    """
    print("Filtering relevant sections...")
    allowed_sections = {"Module 1","Module 2","Module 3","Module 4","Module 5","Module 6", "Module 7","Module 8","Module Intro","Chapter 1", "Chapter 2", "Chapter 3", "Chapter 4", "Chapter 5", "Module Outro"}
    sections = [section for section in page.children if section.type == "SECTION" and section.name in allowed_sections]
    print(f"Found {len(sections)} relevant sections out of {len(page.children)}")
    return sections

def extract_lessons(valid_sections, module_number):
    """
    Groups frames into lessons based on section/subsection names.
    Lessons are identified using a chapter and lesson numbering scheme.
    """
    lesson_groups = defaultdict(list)
    lesson_section_pattern = re.compile(r"Lesson (\d+)", re.IGNORECASE)
    
    for section in valid_sections:
        print(f"\n➡️ Processing section: {section.name}")
        section_name_lower = section.name.lower()
        
        # Handle module intro/outro sections differently
        if section_name_lower in ["module intro", "module outro"]:
            chapter_num = -1 if "outro" in section_name_lower else 0
            lesson_num = 0 if "intro" in section_name_lower else -1
            lesson_id = f"{module_number}.{chapter_num}.{lesson_num}"
            print(f"  🔹 Found lesson: {lesson_id}")
            for frame in section.children:
                if frame.type == "FRAME":
                    lesson_groups[lesson_id].append(frame)
            continue

        if section_name_lower in ["module 1"]:
            for subsection in section.children:
                sub_name = subsection.name.lower()
                chapter_num = 0
                lesson_num = -1 if "preface" in sub_name else 0
                lesson_id = f"{module_number}.{chapter_num}.{lesson_num}"
                print(f"  🔹 Found lesson: {lesson_id}")
                for frame in subsection.children:
                    if frame.type == "FRAME":
                        lesson_groups[lesson_id].append(frame)
                continue
            
        # Determine the chapter number from section
        chapter_match = re.search(r"Chapter (\d+)", section.name)
        if chapter_match:
            chapter_num = int(chapter_match.group(1))
        elif section.name.lower() == "module intro":
            chapter_num = 0
        elif section.name.lower() == "module outro" or section.name.lower() == "module preface":
            chapter_num = -1
        else:
            chapter_num = -99
        
        # Process subsections (lesson-specific) within each section
        for subsection in section.children:
            if subsection.type != "SECTION":
                continue
            sub_name = subsection.name.lower()
            if "chapter intro" in sub_name:
                lesson_num = 0
            elif "chapter outro" in sub_name:
                lesson_num = -1
            elif lesson_section_pattern.match(subsection.name):
                lesson_num = int(lesson_section_pattern.match(subsection.name).group(1))
            else:
                continue
            lesson_id = f"{module_number}.{chapter_num}.{lesson_num}"
            print(f"  🔹 Found lesson: {lesson_id}")
            for frame in subsection.children:
                if frame.type == "FRAME":
                    lesson_groups[lesson_id].append(frame)
    return lesson_groups


# ===============================
# Frame Processing
# ===============================
def build_frame_class_map(raw_map):
    """
    Given a dictionary of raw class mappings, attempts to evaluate each class.
    Returns a mapping of frame names to class objects and a set of names that could not be evaluated.
    """
    frame_map = {}
    skipped = set()
    for name, class_name in raw_map.items():
        try:
            frame_map[name] = eval(class_name)
        except NameError:
            skipped.add(name)
    return frame_map, skipped

def get_color_scheme(frame):
    """
    Infer the color_scheme ("dark" or "light") based on the frame background color.
    """
    if hasattr(frame, "backgroundColor") and frame.backgroundColor:
        color = frame.backgroundColor
        r, g, b = color.get("r", 0), color.get("g", 0), color.get("b", 0)
        brightness = 0.2126 * r + 0.7152 * g + 0.0722 * b
        return "dark" if brightness < 0.5 else "light"
    return "unknown"

def process_frame(frame, idx, frame_class_map, skipped_set, missing_templates, errors, chart_errors):
    """
    Process an individual frame using its mapped class.
    If the frame name is not in the mapping or processing fails, record the issue and return None.
    `chart_errors` is a counter list to track suppressed chart failures.
    """
    if frame.name == "photo-horizontal":
        print(f"📸 Found photo-horizontal frame in lesson: {frame.name}")
    if frame.name not in frame_class_map:
        if frame.name not in skipped_set and not frame.name.lower().endswith("_ignore"):
            missing_templates.add(frame.name)
        return None
    cls = frame_class_map[frame.name]
    try:
        segment = cls.from_node(frame).to_content()
        segment["_order_index"] = idx
        
        # Only infer if template didn't already set it (or it’s unknown)
        if segment.get("color_scheme") in (None, "", "unknown"):
            segment["color_scheme"] = get_color_scheme(frame)
        
        return segment

    except Exception as e:
        error_msg = str(e).strip().split("\n")[0]
        key = (frame.name, cls.__name__, error_msg)

        if "Chart" in cls.__name__:
            chart_errors.append(key)  # suppress logging, just count it
        else:
            print(f"❌ Error in frame {frame.name} ({cls.__name__}): {error_msg}")
            errors.add(key)
        return None


def process_lesson(lesson_id, frame_list, frame_class_map, skipped_set, errors, missing_templates, chart_errors):
    """
    Processes all frames for one lesson.
    Reverses the frame list to account for Figma’s visual ordering,
    then sorts the segments by an internal order index.
    """
    segments = []
    reversed_frames = list(reversed(frame_list))
    for idx, frame in enumerate(reversed_frames):
        segment = process_frame(frame, idx, frame_class_map, skipped_set, missing_templates, errors, chart_errors)
        if segment is not None:
            segments.append(segment)
    segments.sort(key=lambda s: s["_order_index"])
    for s in segments:
        s.pop("_order_index", None)
    return segments



# ===============================
# Unicode Cleaning & File Output
# ===============================
def clean_unicode(text):
    """
    Remove control characters (except newline and tab) and normalize whitespace.
    """
    if not isinstance(text, str):
        return text
    cleaned = ''.join(
        c if unicodedata.category(c)[0] != 'C' or c in '\n\t' else ' '
        for c in text
    )
    return ' '.join(cleaned.split())

def recursively_clean(data):
    """
    Recursively cleans all string values within nested dicts and lists.
    """
    if isinstance(data, dict):
        return {k: recursively_clean(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [recursively_clean(v) for v in data]
    elif isinstance(data, str):
        return clean_unicode(data)
    else:
        return data

def write_lesson_output(lesson_id, segments, output_dir):
    """
    Write the cleaned lesson output to a JSON file.
    """
    cleaned_segments = recursively_clean(segments)
    lesson_output = {"id": lesson_id, "segments": cleaned_segments}
    filename = f"{lesson_id}.json"
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, filename), "w", encoding="utf-8") as f:
        json.dump(lesson_output, f, indent=2, sort_keys=False, ensure_ascii=False)
    print(f"✅ Exported lesson {lesson_id}")


# ===============================
# Photobank & Next-Lesson Mapping (this section updated has new content)
# ===============================


def post_process_images(segments, lesson_id):
    root_file_path = "https://sehseadata.blob.core.windows.net/images/Modules"
    imagekit_file_path = "https://sehseadata.blob.core.windows.net/images/Photos"
    module_num = lesson_id.split(".")[0]
    module_name = f"module_{module_num}".capitalize()

    def update_image(image, allow_random=True):
        if "src" in image:
            try:
                original = image["src"].replace(":", "_").strip()
                if allow_random and module_num in ["4", "7", "8"]:
                    random_image = random.choice(PHOTOBANK_IMAGES) if PHOTOBANK_IMAGES else original
                    image["src"] = f"{imagekit_file_path}/{random_image}"
                    image["caption"] = ""
                else:
                    image["src"] = f"{root_file_path}/{module_name}/{original}.webp"
            except:
                image["src"] = f"{imagekit_file_path}/{random.choice(PHOTOBANK_IMAGES)}"
                image["caption"] = ""

    def update_data(data, allow_random=True):
        if isinstance(data, dict):
            if "image" in data and isinstance(data["image"], dict):
                update_image(data["image"], allow_random=allow_random)
            for value in data.values():
                update_data(value, allow_random=allow_random)
        elif isinstance(data, list):
            for item in data:
                update_data(item, allow_random=allow_random)

    for seg in segments:
        allow_random = not (module_num in ["4", "7", "8"] and seg.get("template_id") == "infographic")
        update_data(seg, allow_random=allow_random)

    return segments


    
# ===============================
# Refactor Global Maps Loader: Only Use 2 Files
# ===============================

import pandas as pd
import json
from collections import defaultdict
import re

def strip_html_tags(text):
    return re.sub(r'<[^>]*>', '', text or "")

def build_lesson_metadata_only(module_json_path):
    """
    Rebuilds lesson metadata using only module_structure.json
    Returns: LESSON_METADATA, CONCEPTS_MAP, RESOURCES_MAP
    """
    with open(module_json_path, "r", encoding="utf-8") as f:
        structure = json.load(f)

    lesson_meta = {}
    for module in structure.get("modules", []):
        for chapter in module.get("chapters", []):
            chapter_id = chapter.get("id")
            chapter_image = chapter.get("image", {})
            chapter_desc = strip_html_tags(chapter.get("description", ""))
            for lesson in chapter.get("lessons", []):
                lesson_id = lesson.get("id")
                if lesson_id:
                    lesson_meta[lesson_id] = {
                        "title": strip_html_tags(lesson.get("title", "")),
                        "image": lesson.get("image", {}).get("src") or chapter_image.get("src", ""),
                        "frame": lesson.get("frame", {}).get("src") or chapter_image.get("src", ""),
                        "caption": lesson.get("image", {}).get("caption") or chapter_image.get("caption", ""),
                        "description": lesson.get("description") or chapter_desc,
                        "progress": lesson.get("progress", "not_started")
                    }

    # Load csv if available to build CONCEPTS_MAP and RESOURCES_MAP
    concepts_map = defaultdict(list)
    resources_map = defaultdict(list)

    try:
        lessons_df = pd.read_csv("../02_Inputs/data/sea-lessons.csv")
        for _, row in lessons_df.iterrows():
            lesson_id = str(row.get("Lesson Number", "")).strip()
            key_concept = row.get("Key Concept", "")
            concept_def = row.get("Concept Definition", "")
        
            # Replace NaN with empty strings
            if pd.isna(key_concept):
                key_concept = ""
            if pd.isna(concept_def):
                concept_def = ""
        
            concept = {
                "title": key_concept,
                "body": concept_def,
                "source": ""
            }
            concepts_map[lesson_id].append(concept)

            # Build resource image src like: https://ik.imagekit.io/seacademy/Resources/Module_1/resource-3211.webp
            module_num = lesson_id.split(".")[0]
            try:
                numeric_id = int(row["Resource ID"])#lesson_id.replace(".", "")+str(row["Resource Number"])
                resource_img = f"https://sehseadata.blob.core.windows.net/images/Resources/Module_{module_num}/resource-{numeric_id}.webp"
    
                resource = {
                    "image": {
                        "src": resource_img,
                        "caption": ""
                    },
                    "href": row["Resource Link"],
                    "text": row["Resource Title"],
                    "cta": "Click to download"
                }
                resources_map[lesson_id].append(resource)
            except:
                pass#resource is missing
    except Exception as e:
        print(f"⚠️ Could not load concepts/resources from csv: {e}")

    
    return lesson_meta, dict(concepts_map), dict(resources_map)

def build_lesson_next_map(module_structure_path):
    """
    Extracts lesson IDs in order from module_structure.json and creates a mapping
    from each lesson id to the next lesson id.
    """
    with open(module_structure_path, "r", encoding="utf-8") as f:
        structure = json.load(f)
    lessons = []
    for module in structure.get("modules", []):
        for chapter in module.get("chapters", []):
            for lesson in chapter.get("lessons", []):
                lesson_id = lesson.get("id")
                if lesson_id:
                    lessons.append(lesson_id)
    lesson_map = {lesson_id: lessons[i+1] if i < len(lessons) - 1 else lesson_id 
                  for i, lesson_id in enumerate(lessons)}
    return lesson_map


import pandas as pd

def build_quiz_object(module_number: int) -> dict:
    """
    Builds a quiz segment object for a given module by reading the corresponding sheet in SEA Quizzes.xlsx.
    Supports multiple correct answers in the form '1,2,4'.
    """
    path = "../02_Inputs/SEA Quizzes.xlsx"
    try:
        df = pd.read_excel(path, sheet_name=f"Module {module_number}")
    except Exception as e:
        print(f"⚠️ Quiz sheet for Module {module_number} not found: {e}")
        return {}

    df = df.dropna(subset=["Question Text"])

    questions = []
    for i, row in df.iterrows():
        # Prepare answer options
        options = [
            {"id": j, "value": str(row[col]).strip()}
            for j, col in enumerate(["Option 1", "Option 2", "Option 3", "Option 4"], start=1)
            if pd.notna(row[col])
        ]

        # Parse correct answer(s)
        raw_correct = str(row.get("Correct", "")).strip()
        try:
            correct_answers = [int(x) for x in raw_correct.split(",") if x.strip().isdigit()]
        except:
            print(f"⚠️ Invalid correct answer format for question {i + 1} in Module {module_number}: {raw_correct}")
            continue

        if not correct_answers:
            continue

        question = {
            "id": i + 1,
            "prompt": str(row["Question Text"]).strip(),
            "multiple": len(correct_answers) > 1,
            "options": options,
            "solution": correct_answers,
            "messages": {
                "correct": {
                    "title": "Correct",
                    "body": str(row.get("Answer Text", "")).strip()
                },
                "wrong": {
                    "title": "Incorrect",
                    "body": str(row.get("Answer Text", "")).strip()
                }
            }
        }
        questions.append(question)

    if not questions:
        return {}

    return {
        "template_id": "scored_quiz",
        "color_scheme": "dark",
        "content": {
            "title": f"Scored quiz <br/><strong>Module {module_number}</strong>",
            "intro": "This quiz covers all the lessons in this module. You must achieve 80% to pass. Good luck!",
            "final": True, ##this should be based on the quiz type, set to True for now just because there are only module-end quizzes
            "labels": {
                "correct": "Correct",
                "question": "Question",
                "result": "Quiz Result",
                "score": "Your score",
                "passingScore": "Minimum passing score",
                "startButton": "Start Quiz",
                "nextButton": "Next",
                "submitButton": "Submit",
                "retryButton": "Replay",
                "passed": "Passed",
                "failed": "Not passed"
            },
            "passingScore": 0.8,
            "questions": questions
        }
    }



def add_list_of_lessons(segments, lesson_id, lesson_metadata):
    """
    Appends a 'list_of_lessons' segment to chapter intro lessons (e.g. '1.1.0').
    Excludes module intro/outro (e.g. '1.0.0', '1.-1.0').
    """
    chapter_intro_pattern = re.compile(r'^(\d+)\.(\d+)\.0$')
    match = chapter_intro_pattern.match(lesson_id)
    if not match:
        return segments  # Not a chapter intro

    module_num, chapter_num = match.groups()
    if chapter_num in ["0", "-1"]:
        return segments  # Skip module intro/outro

    list_items = []
    for lid, data in lesson_metadata.items():
        if not lid.startswith(f"{module_num}.{chapter_num}."):
            continue
        if lid == lesson_id or lid.endswith(".0") or lid.endswith(".-1"):
            continue  # skip intro/outro and self
        list_items.append({
            "title": data.get("title", ""),
            "lessonId": lid,
            "type": "lesson",
            "description": data.get("description", ""),
            "progress": data.get("progress", "not_started"),
            "cta": "Go to the lesson",
            "image": {"src":data.get("frame", ""),
                      "caption":data.get("caption","")}
        })

    if list_items:
        list_of_lessons_segment = {
            "template_id": "list_of_lessons",
            "color_scheme": "dark",
            "content": {
                "title": "What's next in this chapter?",
                "lessons": list_items
            }
        }
        segments.append(list_of_lessons_segment)

    return segments

# Example Usage (Replace path with actual if needed):
MODULE_STRUCTURE_PATH = "../03_Outputs/SEA_Modules/en/module_structure.json"
LESSON_METADATA, CONCEPTS_MAP, RESOURCES_MAP = build_lesson_metadata_only(MODULE_STRUCTURE_PATH)
LESSON_NEXT_MAP = build_lesson_next_map(MODULE_STRUCTURE_PATH)

# Load once globally
with open(MODULE_STRUCTURE_PATH, "r", encoding="utf-8") as f:
    MODULE_STRUCTURE = json.load(f)

def clip_to_first_sentence(text):
    match = re.search(r'([^.?!]+[.?!])', text)
    if match:
        return match.group(1).strip()
    else:
        return text.strip()

def add_post_segments(segments, lesson_id):
    if segments is None:
        print(f"⚠️ Warning: segments for lesson {lesson_id} is None, skipping post-processing.")
        return []

    # Only for normal lessons (not intros .0 or outros .-1)
    if not (lesson_id.endswith(".0") or lesson_id.endswith(".-1")):
    
        # --- Insert key_concepts (3rd segment) ---
        raw_concepts = CONCEPTS_MAP.get(lesson_id, []) or []
        concept_items = []
        for c in raw_concepts:
            title = str(c.get("title", "")).strip()
            body  = str(c.get("body",  "")).strip()
            if title and body:
                concept_items.append({"title": title, "body": body, "source": c.get("source", "")})
            if len(concept_items) == 3:
                break
    
        if concept_items:
            key_concepts = {
                "template_id": "key_concepts",
                "color_scheme": "dark",
                "content": {
                    "title": "Key Concepts",
                    "intro": "Explore foundational ideas from this lesson.",
                    "concepts": concept_items  # matches KeyConcepts schema :contentReference[oaicite:2]{index=2}
                }
            }
            segments.insert(min(2, len(segments)), key_concepts)  # 3rd position (0-based)
    
        # --- Insert key_resources (before connection_next) ---
        raw_resources = RESOURCES_MAP.get(lesson_id, []) or []
        resource_items = []
        for r in raw_resources:
            href = str(r.get("href", "")).strip()
            text = str(r.get("text", "")).strip()
            img  = r.get("image", {}) or {}
    
            # skip empties / NaNs
            if not href or not text:
                continue
    
            # best-effort image fix if helper exists
            if "update_image_path" in globals() and callable(update_image_path) and img:
                try:
                    update_image_path(img)
                except Exception:
                    pass
    
            # guard missing src
            if "src" not in img or not str(img.get("src", "")).strip():
                img = {"src": "", "caption": ""}
    
            resource_items.append({
                "image": img,
                "href": href,
                "text": text,
                "cta": r.get("cta", "Click to download")
            })
            if len(resource_items) == 3:
                break
    
        if resource_items:
            key_resources = {
                "template_id": "key_resources",
                "color_scheme": "dark",
                "content": {
                    "title": "Key Resources",
                    "resources": resource_items  # matches KeyResources schema :contentReference[oaicite:3]{index=3}
                }
            }
            segments.append(key_resources)

    
    if not LESSON_NEXT_MAP:
        build_lesson_next_map()

    next_lesson_id = LESSON_NEXT_MAP.get(lesson_id)
    if not next_lesson_id:
        return segments

    next_meta = LESSON_METADATA.get(next_lesson_id, {})
    module_num = lesson_id.split(".")[0]
    next_module_num = next_lesson_id.split(".")[0]
    next_chap_num = next_lesson_id.split(".")[1]
    next_chap_id = f"{next_module_num}.{next_chap_num}"

    image = {
        "src": f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Modules/module{next_module_num}.webp",
        "caption": ""
    }

    connection_next = {
        "template_id": "connection_next",
        "color_scheme": "dark",
        "content": {
            "image": image,
            "intro": "Next up",
            "title": next_meta.get("title", ""),
            "cta": "Continue learning",#next_meta.get("description", ""),
            "nextLessonId": next_lesson_id
        }
    }

    is_outro = any(s.get("template_id") in ["chapter_outro", "module_outro"] for s in segments)
    has_lesson_cover = any(s.get("template_id") == "lesson_cover" for s in segments)

    if is_outro and not has_lesson_cover:
        for seg in segments:
            if seg.get("template_id") in ["chapter_outro", "module_outro"]:
                if "nextBlock" in seg["content"]:
                    next_block = seg["content"]["nextBlock"]

                    # Default values
                    block_intro = "Continue"
                    block_title = next_meta.get("title", "")
                    block_desc = next_meta.get("description", "")
                    button_cta = "Next chapter"

                    if next_lesson_id.endswith(".-1.0"):
                        button_cta = "Take the quiz"
                        block_intro = f"Module {next_module_num}"
                        block_title = f"Module {next_module_num} Quiz"
                        for mod in MODULE_STRUCTURE["modules"]:
                            if str(mod["id"]) == next_module_num:
                                block_desc = mod.get("description", block_desc)
                                image["src"] = f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Modules/module{next_module_num}.webp"
                                break

                    elif seg["template_id"] == "module_outro":
                        button_cta = "Next module"
                        for mod in MODULE_STRUCTURE["modules"]:
                            if str(mod["id"]) == next_module_num:
                                block_title = mod.get("title", block_title)
                                block_desc = clip_to_first_sentence(mod.get("description", block_desc))
                                block_intro = f"Module {next_module_num}"
                                image["src"] = f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Modules/module{next_module_num}.webp"
                                break

                    else:  # chapter_outro
                        button_cta = "Next chapter"
                        for mod in MODULE_STRUCTURE["modules"]:
                            if str(mod["id"]) == next_module_num:
                                for chap in mod.get("chapters", []):
                                    if str(chap["id"]) == next_chap_id:
                                        block_title = chap.get("title", block_title)
                                        block_desc = clip_to_first_sentence(chap.get("description", block_desc))
                                        block_intro = "Continue to"
                                        image["src"] = f"https://sehseadata.blob.core.windows.net/images/HeaderImages/Chapters/chapter{next_chap_id}.webp"
                                        break

                    # Update the nextBlock
                    next_block.update({
                        "image": image,
                        "nextBlockId": next_lesson_id,
                        "buttonCta": button_cta,
                        "title": block_title,
                        "cta": "",#clip_to_first_sentence(block_desc),
                        "intro": block_intro
                    })
    else:
        segments.append(connection_next)

    return segments



from collections import defaultdict

def track_lesson_frame_order_issues(lesson_groups):
    """
    Populates ORDER_ISSUE_CATEGORIES with lists of lesson_ids violating structural expectations.
    """
    valid_cover_prefixes = (
        "lesson_cover", "lesson_part_cover", "lesson_subpart_cover",
        "chapter_cover", "module_cover","chapter_outro","module_outro"
    )

    for lesson_id, frames in lesson_groups.items():
        frame_names = [f.name for f in reversed(frames)]

        if not frame_names:
            ORDER_ISSUE_CATEGORIES["empty"].append(lesson_id)
            continue

        if any(name == "embed" for name in frame_names):
            ORDER_ISSUE_CATEGORIES["has_embed"].append(lesson_id)

        is_normal_lesson = not lesson_id.endswith(".0") and not lesson_id.endswith(".-1")

        if not any(frame_names[0].startswith(prefix) for prefix in valid_cover_prefixes):
            ORDER_ISSUE_CATEGORIES["missing_cover"].append(lesson_id)

        if is_normal_lesson and len(frame_names) > 1 and frame_names[1] != "text":
            ORDER_ISSUE_CATEGORIES["wrong_second"].append(lesson_id)

        if is_normal_lesson and not frame_names[-1].startswith("key_takeaways"):
            ORDER_ISSUE_CATEGORIES["missing_key_takeaways"].append(lesson_id)

def print_order_issue_summary():
    print("\n📋 Frame Order Check Summary:")
    for category, lesson_ids in ORDER_ISSUE_CATEGORIES.items():
        if lesson_ids:
            label = {
                "missing_cover": "❌ Missing Cover Frame",
                "wrong_second": "⚠️ Second Frame Not 'text'",
                "missing_key_takeaways": "⚠️ Missing Final Key Takeaways",
                "has_embed": "🔗 Contains 'embed' Frame",

            }.get(category, category)
            print(f"\n{label} ({len(lesson_ids)} lessons):")
            for lid in sorted(lesson_ids):
                print(f"  - {lid}")


# ===============================
# Export & Summary Functions
# ===============================
def export_lessons(lesson_groups, frame_class_map, skipped_templates, output_base_dir):
    """
    Processes each lesson into segments, postprocesses them, and writes each as a JSON file.
    Suppresses Chart errors during processing and summarizes them at the end.
    """
    os.makedirs(output_base_dir, exist_ok=True)
    skipped_set = set(skipped_templates)
    errors = set()
    missing_templates = set()
    chart_errors = []  # Collect chart-specific errors for summary

    for lesson_id, frame_list in lesson_groups.items():
        segments = process_lesson(
            lesson_id,
            frame_list,
            frame_class_map,
            skipped_set,
            errors,
            missing_templates,
            chart_errors  # <-- added here
        )

        segments = post_process_images(segments, lesson_id)
        segments = add_list_of_lessons(segments, lesson_id, LESSON_METADATA)
        segments = add_post_segments(segments, lesson_id)
        segments = tag_ai_keywords_in_text_segments(segments)
        write_lesson_output(lesson_id, segments, output_base_dir)

    # Infer module number from the output path (e.g. ".../Module_4")
    match = re.search(r"Module[_ ](\d+)", output_base_dir, re.IGNORECASE)
    module_number = int(match.group(1)) if match else None

    if module_number is not None:
        quiz_segment = build_quiz_object(module_number)
        if quiz_segment:
            quiz_lesson_id = f"{module_number}.-1.0"
            
            ##connection_next is added to quiz for now. Hopefully this can be removed once the logic is in place for the quiz to allow advancing once complete
            segments = post_process_images([quiz_segment], quiz_lesson_id)
            segments = add_post_segments(segments, quiz_lesson_id)
            
            write_lesson_output(quiz_lesson_id, segments, output_base_dir)


    # ✅ Print summary of suppressed chart errors
    if chart_errors:
        print(f"\n📊 Skipped {len(chart_errors)} chart frames due to missing or invalid data.")

    return errors, missing_templates


def print_summary(errors, skipped_templates, missing_runtime_templates, output_dir, lesson_count):
    print(f"\nFinished. {lesson_count} lessons processed from filtered sections. Output in: {output_dir}")
    if errors:
        print(f"\n⚠️ Encountered {len(errors)} unique errors during frame parsing:")
        for frame_name, class_name, error_msg in sorted(errors):
            print(f" - [{class_name}] {frame_name}: {error_msg}")
    if skipped_templates:
        print("\n⚠️ Skipped templates due to missing class definitions (initial map):")
        for template in sorted(skipped_templates):
            print(f" - {template}")
    if missing_runtime_templates:
        print("\n⚠️ Skipped templates not found in class map (encountered during lessons):")
        for template in sorted(missing_runtime_templates):
            print(f" - {template}")




# ===============================
# Pipeline Orchestration
# ===============================
def process_pages(page_list, loader_func, raw_frame_class_map):
    """
    Iterates through a list of page names, loads each page using the provided loader function,
    extracts lessons and exports them.
    """
    for page_name in page_list:
        print(f"\n========================\n📄 Processing {page_name}\n========================")
        page = loader_func(page_name)
        valid_sections = get_valid_sections(page)
        frame_class_map, skipped_templates = build_frame_class_map(raw_frame_class_map)
        module_number_match = re.search(r"\d+", page_name)
        if not module_number_match:
            raise ValueError(f"Module number not found in page name: {page_name}")
        module_number = int(module_number_match.group())
        lesson_groups = extract_lessons(valid_sections, module_number)
        print(f"\nGrouped into {len(lesson_groups)} lessons.")
        track_lesson_frame_order_issues(lesson_groups)
        output_dir = os.path.join(OUTPUT_DIR_BASE, page_name.replace(" ", "_"))
        errors, missing_runtime_templates = export_lessons(lesson_groups, frame_class_map, skipped_templates, output_dir)
        print_summary(errors, skipped_templates, missing_runtime_templates, output_dir, len(lesson_groups))

def run_pipeline(page_list1, page_list2, redownload=False):
    """
    Main pipeline function that optionally re-downloads the Figma documents and processes
    pages from both document versions.
    """
    if redownload:
        print("Downloading Figma document1...")
        FIGMA_DOCUMENT1 = Document.from_file_key(FIGMA_DOCUMENT_ID1)
        save_figma_to_json(FIGMA_DOCUMENT1, "figma_document1.json")

        print("Downloading Figma document2...")
        FIGMA_DOCUMENT2 = Document.from_file_key(FIGMA_DOCUMENT_ID2)
        save_figma_to_json(FIGMA_DOCUMENT2, "figma_document2.json")
    
    # Note: The commented class map entries below are preserved for later re-enabling:
    raw_frame_class_map = {
        "lesson_cover": "LessonCover",
        "module_cover": "ModuleCover",
        "case_study_cover":"CaseStudyCover",
        "learning_objectives": "LearningObjectives",
        "photo-vertical": "PhotoVertical",
        "photo-horizontal": "PhotoHorizontal",
        "photo-full-height": "PhotoFullHeight",
        "video": "Video",
        "text": "ModuleText",  
        "lesson_subpart_cover": "LessonSubpartCover",
        "lesson_part_cover": "LessonPartCover",
        "chart":"Chart",#_folder",
        "infographic":"Infographic",
        "chapter_outro": "ChapterOutro",
        "module_outro": "ModuleOutro",
        "key_takeaways": "KeyTakeaways",    
        "chapter_cover": "ChapterCover",
        "embed":"Embed",
        "poll":"Poll"
    }

    chart_meta_map = frames.load_chart_metadata_from_tracker_xlsx("../02_Inputs/Charts/Metadata/Charts Tracker.xlsx")
    frames.set_chart_metadata_map(chart_meta_map)
    print(f"✅ Loaded chart metadata for {len(chart_meta_map)} charts")
    
    process_pages(page_list1, lambda name: load_cached_figma_page("figma_document1.json", name), raw_frame_class_map)
    process_pages(page_list2, lambda name: load_cached_figma_page("figma_document2.json", name), raw_frame_class_map)
    print_order_issue_summary()

# Run pipeline
run_pipeline(page_list1, page_list2, redownload=False)


In [ ]:
import pandas as pd
import json
from pathlib import Path

def insert_ai_avatars():
    # Load the Excel file
    df = pd.read_excel("../02_Inputs/SEA AI Avatars.xlsx")
    df = df.dropna(subset=["Lesson", "Language"])

    for _, row in df.iterrows():
        lesson_id = str(row["Lesson"]).strip()       # e.g., "1.1.1"
        language = str(row["Language"]).strip()      # e.g., "en"

        # Extract module number and lesson number
        parts = lesson_id.split(".")
        if len(parts) != 3:
            print(f"❌ Skipping invalid lesson ID: {lesson_id}")
            continue

        module_number = parts[0]
        lesson_number = ".".join(parts)

        # Construct path to JSON file
        json_path = Path(f"../03_Outputs/SEA_Modules/{language}/Module_{module_number}/{lesson_number}.json")
        if not json_path.exists():
            print(f"⚠️ File not found: {json_path}")
            continue

        # Load existing JSON
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Construct the video segment with updated paths
        video_segment = {
            "template_id": "video",
            "color_scheme": "light",
            "size": "full",
            "content": {
                "src": f"https://sehseadata.blob.core.windows.net/images/AIAvatars/Videos/{lesson_id}-{language}.mp4",
                "poster": f"https://sehseadata.blob.core.windows.net/images/AIAvatars/Photos/{lesson_id}-{language}.webp"
            }
        }

        segments = data.get("segments", [])

        # Check if a matching video segment already exists at index 2
        if len(segments) > 2:
            existing_segment = segments[2]
            if (
                existing_segment.get("template_id") == "video" and
                existing_segment.get("content", {}).get("src") == video_segment["content"]["src"]
            ):
                print(f"⏭️ Skipping (already exists): {json_path}")
                continue

        # Insert at index 2 (3rd position) without overwriting
        while len(segments) < 2:
            segments.append({})
        segments = segments[:2] + [video_segment] + segments[2:]

        # Write back to file
        data["segments"] = segments
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"✅ Appended avatar segment to: {json_path}")

insert_ai_avatars()


In [ ]:
def collect_lesson_ids_from_folder(root_folder):
    """
    Walks through all subdirectories of root_folder, finds .json files,
    strips the '.json' extension, and returns the list.
    """
    lesson_ids = []
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            if filename.endswith(".json") and filename != "module_structure.json":
                lesson_id = filename[:-5]  # Remove ".json"
                lesson_ids.append(lesson_id)
    return lesson_ids

all_lessons = collect_lesson_ids_from_folder("../03_Outputs/SEA_Modules/en")
print("\n🗂️ All lesson IDs found:")
print(all_lessons)

open("../03_Outputs/all_lesson_ids.txt", "w").write(str(all_lessons))


In [ ]:
import os
import json
import math
import chardet

def find_invalid_numbers(obj, path="$"):
    invalids = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            invalids.extend(find_invalid_numbers(value, f"{path}.{key}"))
    elif isinstance(obj, list):
        for idx, value in enumerate(obj):
            invalids.extend(find_invalid_numbers(value, f"{path}[{idx}]"))
    elif isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            invalids.append((path, obj))
    return invalids

def sanitize_invalid_numbers(obj):
    if isinstance(obj, dict):
        return {k: sanitize_invalid_numbers(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize_invalid_numbers(v) for v in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return ""
        return obj
    return obj

def detect_encoding(filepath):
    with open(filepath, 'rb') as f:
        raw = f.read(10000)
        result = chardet.detect(raw)
        return result["encoding"] or "utf-8"

def validate_and_sanitize_json_files(root_folder):
    sanitized_files = []
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            if filename.lower().endswith('.json'):
                file_path = os.path.join(dirpath, filename)
                try:
                    # Try reading with utf-8 first
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            data = json.load(f)
                    except UnicodeDecodeError:
                        print(f"⚠️  Unicode decode failed for {file_path}, trying latin-1 as fallback...")
                        with open(file_path, 'r', encoding='latin-1') as f:
                            data = json.load(f)

                    invalid_entries = find_invalid_numbers(data)
                    if invalid_entries:
                        msgs = "; ".join(f"{p}={v}" for p, v in invalid_entries)
                        sanitized_files.append((file_path, msgs))
                        cleaned = sanitize_invalid_numbers(data)
                        with open(file_path, 'w', encoding='utf-8') as f:
                            json.dump(cleaned, f, ensure_ascii=False, indent=2)
                except Exception as e:
                    print(f"⚠️  Failed to process {file_path}: {e}")
    return sanitized_files


# === Run for all language folders ===
languages = ['en', 'es', 'pt', 'fr', 'ar','ru','zh']
for lang in languages:
    folder_path = os.path.join('../03_Outputs', 'SEA_Modules', lang)
    print(f"🔍 Scanning and sanitizing JSON files in {folder_path}...")
    results = validate_and_sanitize_json_files(folder_path)
    if results:
        print(f"✅ Sanitized files in [{lang}]:")
        for path, details in results:
            print(f" - {path}: replaced values {details}")
    else:
        print(f"✅ No invalid numeric values found in [{lang}]; all files are clean.")


In [ ]:
import os
import json
import math
import re
from html import unescape
from bs4 import BeautifulSoup

# --- Configuration ---

TEMPLATE_HEIGHT_RULES = {
    # Fixed-height templates
    "subtitle": {"type": "fixed", "units": 1.5},
    "subtitle_small": {"type": "fixed", "units": 1.0},
    "quote_large_with_name": {"type": "fixed", "units": 3.5},
    "quote_large_without_name": {"type": "fixed", "units": 3.5},
    "quote_small_with_name": {"type": "fixed", "units": 2.0},
    "quote_small_without_name": {"type": "fixed", "units": 2.0},
    "kpi_highlight_large": {"type": "fixed", "units": 3.0},
    "kpi_highlight_medium": {"type": "fixed", "units": 2.0},

    # Text-length-dependent templates
    "paragraph_large": {"type": "dynamic", "per_line": 0.5},
    "paragraph_medium": {"type": "dynamic", "per_line": 0.5},
    "paragraph_small": {"type": "dynamic", "per_line": 0.5},
    "bullet_point": {"type": "dynamic", "per_line": 0.5},
    "bullet_point_with_highlight": {"type": "dynamic", "per_line": 0.5},
    "bullet_point_with_number": {"type": "dynamic", "per_line": 0.5},
}

MAX_SEGMENT_UNIT_THRESHOLD = 12
CHARS_PER_LINE = 80

# --- Helper Functions ---

def strip_html(text):
    return BeautifulSoup(unescape(text), "html.parser").get_text()

def estimate_text_units(template_id, raw_text):
    rule = TEMPLATE_HEIGHT_RULES.get(template_id)
    if not rule or not raw_text.strip():
        return 0.0
    text = strip_html(raw_text)
    if rule["type"] == "fixed":
        return rule["units"]
    elif rule["type"] == "dynamic":
        line_count = math.ceil(len(text) / CHARS_PER_LINE)
        return line_count * rule["per_line"]
    return 0.0

def estimate_segment_units(text_elements):
    total_units = 0.0
    for elem in text_elements:
        tid = elem.get("template_id")
        content = elem.get("content", {})
        raw_text = content.get("text") or content.get("quote") or content.get("title") or content.get("body") or ""
        total_units += estimate_text_units(tid, raw_text)
    return total_units

def extract_lesson_number(path):
    """Extracts lesson number like '1.1.1' and returns tuple of ints for sorting."""
    match = re.search(r'/(\d+)\.(\d+)\.(\d+)\.json$', path)
    if match:
        return tuple(map(int, match.groups()))
    return (999, 999, 999)

# --- Main Function ---

def flag_overflow_segments(root_folder):
    flagged_segments = []
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            if filename.lower().endswith(".json"):
                file_path = os.path.join(dirpath, filename)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        data = json.load(f)

                    for idx, segment in enumerate(data.get("segments", [])):
                        if segment.get("template_id") == "text":
                            text_elements = segment.get("content", {}).get("text_elements", [])
                            estimated_units = estimate_segment_units(text_elements)
                            if estimated_units > MAX_SEGMENT_UNIT_THRESHOLD:
                                excerpt = ""
                                for el in text_elements:
                                    text = el.get("content", {}).get("text") or el.get("content", {}).get("quote") or el.get("content", {}).get("title") or ""
                                    if text:
                                        excerpt = strip_html(text)[:100]
                                        break
                                flagged_segments.append({
                                    "file": file_path,
                                    "segment_index": idx,
                                    "estimated_units": round(estimated_units, 2),
                                    "excerpt": excerpt
                                })

                except Exception as e:
                    print(f"⚠️ Failed to process {file_path}: {e}")
    return sorted(flagged_segments, key=lambda x: extract_lesson_number(x["file"]))

# --- Run and Print ---

if __name__ == "__main__":
    root_folder = "../03_Outputs/SEA_Modules/en"  # or whatever your path is
    flagged = flag_overflow_segments(root_folder)
    for item in flagged:
        print(f"{item['file']} | Segment #{item['segment_index']} has {item['estimated_units']} units (max {MAX_SEGMENT_UNIT_THRESHOLD})")
        print(f"   ➤ Excerpt: {item['excerpt']}\n")
